# **Importing Libraries and Configuring GPU** 

We start by importing essential libraries for data manipulation (NumPy, Pandas),
deep learning (TensorFlow/Keras), and model optimization (ReduceLROnPlateau, EarlyStopping, etc.).
We also configure GPU usage for faster training.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

from tqdm.auto import tqdm

from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization
from keras.callbacks import ReduceLROnPlateau
from keras.optimizers import RMSprop

seed = 42
print(tf.config.list_physical_devices('GPU'))

In [ ]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

# **Loading the Dataset**

We load the train and test datasets for digit recognition using pandas.read_csv().

train.csv → Contains labeled digit images (features + labels).

test.csv → Contains only digit images (no labels, for prediction).

In [ ]:
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [ ]:
X = train_df.iloc[:, 1:]
y = train_df.iloc[:, 0]
X.iloc[0][133]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = seed)
X_train = X_train.values.reshape(-1, 28, 28) / 255.0
X_test = X_test.values.reshape(-1, 28, 28) / 255.0

y_train = y_train.values
X_train = np.expand_dims(X_train, axis = -1) 
y_test = y_test.values

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Data Augmentation using ImageDataGenerator
datagen = ImageDataGenerator(
    zca_whitening=False,                  
    rotation_range=13,                    
    zoom_range=0.1,                       
    width_shift_range=0.1,                 
    height_shift_range=0.1,                 
)

# Generate augmented data batches for training
train_gen = datagen.flow(X_train, y_train)

# **Building a Residual Block**

In [ ]:
class ResidualUnit(keras.layers.Layer):
    def __init__(self, filters, strides = 1, activation = keras.activations.relu, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.strides = strides
        self.activation = activation
        self.main_layer = [
            keras.layers.Conv2D(filters, 3, strides = strides, padding = "same", use_bias = False),
            keras.layers.BatchNormalization(),
            keras.layers.Activation("relu"),
            keras.layers.Conv2D(filters, 3, strides = 1, padding = "same", use_bias = False),
            keras.layers.BatchNormalization()
        ]
        self.skip_layers = []
        if strides > 1:
            self.skip_layers = [
                keras.layers.Conv2D(filters, 1, strides = strides, padding = "same", use_bias = False),
                keras.layers.BatchNormalization(),
            ]

    def call(self, inputs):
        # Forward pass through main path
        Z = inputs
        for layer in self.main_layer:
            Z = layer(Z)

        # Forward pass through skip path (identity if strides=1)
        skip_Z = inputs
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)

        # Add main path and skip path (Residual connection)
        return self.activation(Z + skip_Z)

In [ ]:
# Initialize the Sequential model
model = Sequential()

# First Convolutional Block
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding='Same', activation='relu', input_shape=(28, 28, 1)))
model.add(BatchNormalization())          # Normalize activations to stabilize learning

# Second Convolutional Block
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding='Same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))   # Down-sample feature maps (reduce spatial size)

# Third Convolutional Block
model.add(Conv2D(filters=64, kernel_size=(5, 5), padding='Same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(0.20))

# Fourth Convolutional Block
model.add(Conv2D(filters=64, kernel_size=(5, 5), padding='Same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(0.20))

# Fifth Convolutional Block
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='Same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(0.20))

# Flatten + Fully Connected Layers
model.add(Flatten())                     # Flatten feature maps into a 1D vector
model.add(Dense(256, activation="relu")) # Dense layer with 256 neurons
model.add(BatchNormalization())
model.add(Dropout(0.20))

# Output Layer
model.add(Dense(10, activation="softmax"))  # Final layer: 10 classes (digits 0–9)

In [ ]:
# Define optimizer: RMSprop with tuned hyperparameters
optimizer = RMSprop(
    learning_rate=0.001,   # Step size for weight updates
    rho=0.9,               # Decay factor for moving average of squared gradients
    epsilon=1e-08,         # Small constant to prevent division by zero
    decay=0.0              # Learning rate decay (not used here)
)

# Compile the model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',  # Suitable for integer class labels
    metrics=['accuracy']                     # Track accuracy during training
)

# Early stopping to prevent overfitting
early_stopping = keras.callbacks.EarlyStopping(
    patience=12,            # Stop if validation loss does not improve for 12 epochs
)

In [ ]:
scheduler = ReduceLROnPlateau(monitor = 'val_acc', patience = 3, verbose = 1, factor = 0.5, min_lr = 0.00001)

In [ ]:
def lr_scheduler(epoch, lr):
    if epoch < 50:
        return lr
    else:
        return lr * 0.95

scheduler = keras.callbacks.LearningRateScheduler(lr_scheduler)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, scheduler]
)

In [ ]:
import matplotlib.pyplot as plt

# Plot Training & Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Plot Training & Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
test_df = test_df.values.reshape(-1, 28, 28) / 255.0
predictions = model.predict(test_df)
predicted_labels = np.argmax(predictions, axis = 1)

In [ ]:
print(predicted_labels)

In [ ]:
submission = pd.DataFrame({
    'ImageId': np.arange(1, len(predictions) + 1),
    'Label': predicted_labels
})

print("Saving submission file")
submission.to_csv('submission.csv', index = False)
print("submission.csv saved!")